<a href="https://colab.research.google.com/github/jason-snow58/Tuning-the-Stability-of-a-Disulfide-Stabilized-Phage-VLP-by-Interface-Guided-Capsid-Engineering/blob/main/Tuning_the_Stability_of_a_Disulfide_Stabilized_Phage_VLP_by_Interface_Guided_Capsid_Engineering_Interdimer_interface_residue_contact_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Interdimer interface residue contact detection

Supporting analysis for *Tuning the Stability of a Disulfide-Stabilized Phage VLP by Interface-Guided Capsid Engineering*

---

### What the code does

Inside a live PyMOL session, each selected residue is probed by its side-chain heavy atoms against all heavy atoms of its neighbours, and for each neighbouring residue the closest qualifying atom pair is recorded and drawn as a distance object.

### One setting that changes the data

`INTRA_MODE` decides what counts as "the same unit". The historical runs used `chain`, and because LE, ME and NE all carry PDB chain letter `F` while belonging to three different dimers, every contact between them satisfied both searches and was **written twice, with contradictory labels**. `segi` keeps the two searches disjoint. Both are offered: `chain` to reproduce the historical output, `segi` for a table without the duplicates.

Note also that `contact_group` records a *segment* relationship, not a dimer relationship — this step has no dimer map. Dimer membership is applied downstream, when the maps are drawn.

> **A note on structure.** Unlike the Figure 1 notebook, this step is a *preserved producer*: reading, deciding, drawing and writing happen inside one operation, and it changes PyMOL state directly. It is kept in that shape deliberately, because it reproduces the retained output and restructuring its internals would put that at risk. The phase separation described in the Figure 1 notebook does not apply here, and the notebook does not pretend otherwise.

## Setup

In [ ]:
#@title Setup and input controls { display-mode: "form" }
#@markdown Run this cell first. It detects the environment, installs anything
#@markdown missing, imports everything the notebook needs, and collects the
#@markdown input settings below. Defaults reproduce the published analysis.

# ---- environment -----------------------------------------------------------
try:
    import google.colab            # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import glob
import os
import subprocess
import sys

def ensure(package, module=None):
    """Import a package, installing it first if this is Colab."""
    name = module or package
    try:
        __import__(name)
        return True
    except ImportError:
        if IN_COLAB:
            print(f"installing {package} ...")
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", package],
                           check=True)
            __import__(name)
            return True
        print(f"MISSING: {package}")
        print(f"  conda install -c conda-forge {package}")
        return False


# ---- dependencies ----------------------------------------------------------
# PyMOL is the dependency that differs most between the two environments: in
# Colab the open-source wheel installs cleanly, and locally it is usually
# already present in a conda environment.
READY = ensure("pymol-open-source", "pymol2")
import pymol2
print("PyMOL ready")

# ---- input controls --------------------------------------------------------
#@markdown **Structure.** Leave blank to search `data/` for a `.pdb`.
PDB_FILE = "" #@param {type:"string"}
#@markdown **Residues to probe.** A PyMOL selection expression. The paper's
#@markdown hydrophobic-patch residues are I8, L23 and F116.
SELECTION = "resi 8+23+116" #@param {type:"string"}
#@markdown **Contact cutoff (Å).**
CUTOFF = 4.0 #@param {type:"number"}
#@markdown **PyMOL object name.** This is written into the `source_model`,
#@markdown `target_model` and `*_atom_id` columns, so it is part of the output,
#@markdown not a cosmetic choice.
OBJECT_NAME = "Base_Structure" #@param {type:"string"}
#@markdown **Intradimer test.** `segi` keeps the two searches disjoint. `chain`
#@markdown reproduces the historical invocation, in which chains LE, ME and NE
#@markdown share PDB chain letter `F` and contacts between them are written
#@markdown twice with contradictory labels.
INTRA_MODE = "segi" #@param ["segi", "chain"]
OUTPUT_DIR = "output" #@param {type:"string"}

# ---- helpers ---------------------------------------------------------------
def find_input(description, patterns, search_dirs, allow_multiple=False):
    """Locate an input file, or offer an upload box in Colab.

    Patterns are tried in order, so a specific name wins over a general glob.
    Returns one path unless allow_multiple=True.
    """
    for pattern in patterns:
        hits, seen = [], set()
        for directory in search_dirs:
            # '**' searches subdirectories, which matters because each analysis
            # is committed into its own directory named after the structure.
            found = glob.glob(os.path.join(directory, pattern), recursive=True)
            for path in sorted(found):
                real = os.path.realpath(path)
                if os.path.isfile(path) and real not in seen:
                    seen.add(real)
                    hits.append(path)
        if hits:
            print(f"input: matched {pattern!r}")
            for h in hits:
                print("   ", h)
            if len(hits) > 1 and not allow_multiple:
                print("   using the first; set the variable directly to choose another")
                return hits[0]
            return hits if allow_multiple else hits[0]

    if IN_COLAB:
        from google.colab import files
        print(f"Upload {description}:")
        uploaded = files.upload()
        if not uploaded:
            return None
        names = list(uploaded)
        return names if allow_multiple else names[0]

    print(f"Nothing found for {description}.")
    print("Searched:", ", ".join(search_dirs))
    print("Patterns:", ", ".join(repr(p) for p in patterns))
    return None


def require(value, what):
    if value is None:
        raise SystemExit(f"No input for {what}. See the message above.")
    return value


def deliver(paths):
    """Report finished files, and download them in Colab."""
    paths = [paths] if isinstance(paths, str) else list(paths)
    for p in paths:
        size = os.path.getsize(p) if os.path.exists(p) else 0
        print(f"   {p}  ({size:,} bytes)")
    if IN_COLAB:
        from google.colab import files
        for p in paths:
            files.download(p)

os.makedirs(OUTPUT_DIR, exist_ok=True)
print()
print("Colab" if IN_COLAB else "local Jupyter", "| python", sys.version.split()[0])
print("output directory:", os.path.abspath(OUTPUT_DIR))

## The analysis code

These cells write the analysis package into the working directory, so the notebook is self-contained and nothing has to be fetched. This is the same source that accompanies the manuscript — read it if you want to check the calculation, or run straight past it.

In [ ]:
os.makedirs("capsid", exist_ok=True)
print("package directory ready")

In [ ]:
%%writefile capsid/__init__.py
"""Interface and contact analysis for ssRNA phage capsids.

WHICH PIPELINES ARE SEPARATED, AND WHICH ARE NOT

Only the contact-classification path implements the phase separation in its
call graph:

    decide.analyse(pdb)                  reads the structure; classifies; once
    render.render_analysis(decision)     pure; produces every finished file
    validate.validate_rendering(...)     three derivations compared row by row
    effect.commit_artifacts(...)         transport only; cannot render

``effect`` does not import ``render`` and never receives an analysis record, so
no output path can decide or render once writing has begun. Files are staged
and committed as a set, so a failed run leaves the previous output untouched
rather than a directory holding half of one run and half of another.

The three PyMOL and figure modules -- ``pymol_contacts``, ``calpha_svg`` and
``network_map`` -- are a different shape. Each interleaves reading, deciding,
drawing and writing in a single operation, and each writes files or changes
PyMOL state directly. They are kept that way deliberately, because they
reproduce the retained outputs and restructuring their internals would put that
at risk. The guarantees above describe the classification path only.
"""

import importlib

__all__ = ['artifacts', 'calpha_svg', 'decide', 'effect', 'manifest',
           'network_map', 'palette', 'pymol_contacts', 'records', 'render',
           'validate']


def __getattr__(name):
    """Import submodules on first use.

    Importing the package must not require every optional dependency. A
    notebook that only classifies contacts needs neither matplotlib nor PyMOL,
    and should not be made to install them to say ``import capsid``.
    """
    if name in __all__:
        module = importlib.import_module(f'.{name}', __name__)
        globals()[name] = module
        return module
    raise AttributeError(f'module {__name__!r} has no attribute {name!r}')


def __dir__():
    return sorted(__all__)

In [ ]:
%%writefile capsid/pymol_contacts.py
"""Residue contact detection inside a live PyMOL session.

    PyMOL> run capsid/pymol_contacts.py
    PyMOL> chain_contacts selection="resi 8+23+116", csv_path="contacts.csv"

For every residue in the selection, side-chain heavy atoms are probed against
all heavy atoms of neighbouring residues. For each neighbouring residue that
comes within the cutoff, the closest qualifying atom pair is recorded and drawn
as a PyMOL distance object. Output is a 23-column CSV, one row per contact.

SUBUNITS ARE IDENTIFIED BY SEGMENT, NOT BY CHAIN. The PDB chain column is
degenerate in these biological assemblies: in the AP205 trimer-of-dimers,
chains LE, ME and NE all carry chain letter F while belonging to three different
dimers. Both ``inter_mode`` and ``intra_mode`` therefore default to ``'segi'``,
which keeps the two searches disjoint so that every contact is written exactly
once.

``contact_group`` records a SEGMENT relationship, not a dimer relationship. This
module has no dimer map, so a contact between the two subunits of a single dimer
is reported as ``inter``. Dimer membership is applied downstream, in
network_map.py.

Requires PyMOL. Tested with PyMOL 3.2 (open-source).
"""

import csv
import math

CSV_FIELDS = [
    'contact_group', 'chain_relation', 'segi_relation', 'filter_mode', 'touch',
    'cutoff_angstrom', 'distance_angstrom',
    'source_model', 'source_segi', 'source_chain', 'source_resi', 'source_resn',
    'source_atom', 'source_atom_index', 'source_atom_id',
    'target_model', 'target_segi', 'target_chain', 'target_resi', 'target_resn',
    'target_atom', 'target_atom_index', 'target_atom_id',
]

BACKBONE_ATOMS = frozenset({'N', 'CA', 'C', 'O', 'OXT'})

INTER_DIMER_DASH_COLOR_NAME = 'inter_dimer_dark_blue'
INTER_DIMER_DASH_COLOR_RGB = [0.0, 0.0, 0.35]

# How a target must relate to its source. The inter and intra searches share
# the SAME four combination rules over OPPOSITE base predicates -- inter
# requires the segi/chain to differ, intra requires them to match. The two are
# spelled out separately rather than as one negated table, because negation
# does not distribute the way it appears to: inter-'either' means
# "different_segi OR different_chain", which is not the negation of
# intra-'either'. De Morgan makes that an easy and silent mistake.
COMBINE = {
    'segi':   lambda segi_ok, chain_ok: segi_ok,
    'chain':  lambda segi_ok, chain_ok: chain_ok,
    'either': lambda segi_ok, chain_ok: segi_ok or chain_ok,
    'both':   lambda segi_ok, chain_ok: segi_ok and chain_ok,
}


class SearchMode:
    """The whole difference between the inter and intra searches."""

    def __init__(self, group, want_different, default_filter, dash_color):
        self.group = group                    # 'inter' | 'intra'
        self.want_different = want_different  # inter: differ; intra: match
        self.default_filter = default_filter
        self.dash_color = dash_color          # colour applied to dash objects

    def accepts(self, source, target, filter_mode):
        combine = COMBINE.get(filter_mode)
        if combine is None:
            raise ValueError(
                f"{self.group}_mode must be one of: {', '.join(COMBINE)}")
        if self.want_different:
            segi_ok = target.segi != source.segi
            chain_ok = target.chain != source.chain
        else:
            segi_ok = target.segi == source.segi
            chain_ok = target.chain == source.chain
        return combine(segi_ok, chain_ok)


INTER = SearchMode('inter', True, 'segi', INTER_DIMER_DASH_COLOR_NAME)
INTRA = SearchMode('intra', False, 'segi', None)


# --------------------------------------------------------------------------
# pure helpers -- no PyMOL, unit-testable
# --------------------------------------------------------------------------

def _clean_name(text):
    """PyMOL object names may not contain punctuation."""
    out = str(text)
    for bad in "'\" /\\-:;,":
        out = out.replace(bad, '_')
    return out


def is_sidechain(atom):
    return atom.name not in BACKBONE_ATOMS


def is_hydrogen(atom):
    return atom.symbol.upper() == 'H' or atom.name.upper().startswith('H')


def squared_distance(a, b):
    ax, ay, az = a.coord
    bx, by, bz = b.coord
    return (ax - bx) ** 2 + (ay - by) ** 2 + (az - bz) ** 2


def residue_key(atom):
    return (atom.model, atom.segi, atom.chain, atom.resi, atom.resn)


def atom_id(atom):
    return f'{atom.model}/{atom.segi}/{atom.chain}/{atom.resn}{atom.resi}/{atom.name}`{atom.index}'


def keep_atoms(atoms, sidechain_only=True, exclude_h=True):
    return [a for a in atoms
            if not (sidechain_only and not is_sidechain(a))
            and not (exclude_h and is_hydrogen(a))]


def contact_row(group, filter_mode, cutoff, source_key, target_key,
                source_atom, target_atom, dist2):
    """Build one CSV record. Pure: the same inputs always give the same row."""
    s_model, s_segi, s_chain, s_resi, s_resn = source_key
    t_model, t_segi, t_chain, t_resi, t_resn = target_key
    return {
        'contact_group': group,
        'chain_relation': ('intra_chain' if source_atom.chain == target_atom.chain
                           else 'inter_chain'),
        'segi_relation': ('intra_segi' if source_atom.segi == target_atom.segi
                          else 'inter_segi'),
        'filter_mode': filter_mode,
        'touch': 'yes',
        'cutoff_angstrom': f'{float(cutoff):.3f}',
        'distance_angstrom': f'{math.sqrt(dist2):.3f}',
        'source_model': s_model, 'source_segi': s_segi, 'source_chain': s_chain,
        'source_resi': s_resi, 'source_resn': s_resn,
        'source_atom': source_atom.name, 'source_atom_index': source_atom.index,
        'source_atom_id': atom_id(source_atom),
        'target_model': t_model, 'target_segi': t_segi, 'target_chain': t_chain,
        'target_resi': t_resi, 'target_resn': t_resn,
        'target_atom': target_atom.name, 'target_atom_index': target_atom.index,
        'target_atom_id': atom_id(target_atom),
    }


def group_by_residue(atoms):
    """Index every atom by residue once, so the search does not rescan the
    whole session for each source residue."""
    grouped = {}
    for atom in atoms:
        grouped.setdefault(residue_key(atom), []).append(atom)
    return grouped


def closest_pairs(source_atoms, target_residues, cutoff2, mode, source_key,
                  filter_mode, exclude_h, target_sidechain_only,
                  same_model_only):
    """For each candidate target residue, the single closest qualifying pair.

    Returns {target_key: (dist2, source_atom, target_atom)} for residues that
    have at least one pair inside the cutoff. Ties keep the first pair seen, in
    the atom order PyMOL reported.
    """
    source_model = source_key[0]
    probe = source_atoms[0]
    found = {}
    for target_key, atoms in target_residues.items():
        if target_key == source_key:
            continue
        if same_model_only and target_key[0] != source_model:
            continue
        if not mode.accepts(probe, atoms[0], filter_mode):
            continue
        targets = keep_atoms(atoms, target_sidechain_only, exclude_h)
        if not targets:
            continue
        best = None
        for s_atom in source_atoms:
            for t_atom in targets:
                d2 = squared_distance(s_atom, t_atom)
                if d2 <= cutoff2 and (best is None or d2 < best[0]):
                    best = (d2, s_atom, t_atom)
        if best is not None:
            found[target_key] = best
    return found


def write_contact_csv(csv_path, rows):
    with open(csv_path, 'w', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=CSV_FIELDS)
        writer.writeheader()
        writer.writerows(rows)
    print(f'Wrote {len(rows)} contact rows to: {csv_path}')
    return len(rows)


# --------------------------------------------------------------------------
# the search, shared by both modes
# --------------------------------------------------------------------------

def _search(mode, selection, cutoff, out_prefix, source_sidechain_only,
            target_sidechain_only, exclude_h, same_model_only, filter_mode,
            one_object, verbose, make_selection, selection_name, csv_path):
    from pymol import cmd

    filter_mode = filter_mode or mode.default_filter
    cmd.delete(f'{out_prefix}*')
    if make_selection:
        cmd.delete(selection_name)

    source_model_obj = cmd.get_model(f'byres ({selection})')
    if not source_model_obj.atom:
        print(f'No atoms found for selection: {selection}')
        return []
    all_atoms = cmd.get_model('all').atom
    if not all_atoms:
        print('No atoms found in PyMOL session.')
        return []

    target_residues = group_by_residue(all_atoms)
    source_residues = group_by_residue(source_model_obj.atom)
    print(f'Found {len(source_residues)} source residues.')

    if mode.dash_color:
        cmd.set_color(INTER_DIMER_DASH_COLOR_NAME, INTER_DIMER_DASH_COLOR_RGB)

    rows = []
    selection_atoms = []
    cutoff2 = cutoff ** 2

    for source_key, source_all in sorted(source_residues.items()):
        s_model, s_segi, s_chain, s_resi, s_resn = source_key
        source_atoms = keep_atoms(source_all, source_sidechain_only, exclude_h)
        if not source_atoms:
            if verbose:
                print('Skipping source with no atoms after filters:', s_resn,
                      s_resi, 'model:', s_model, 'segi:', repr(s_segi),
                      'chain:', repr(s_chain))
            continue

        hits = closest_pairs(source_atoms, target_residues, cutoff2, mode,
                             source_key, filter_mode, exclude_h,
                             target_sidechain_only, same_model_only)

        dash_name = _clean_name(out_prefix if one_object else
                                f'{out_prefix}_{s_model}_{s_segi}_{s_chain}_{s_resn}{s_resi}')

        for target_key, (dist2, s_atom, t_atom) in sorted(hits.items()):
            cmd.distance(dash_name,
                         f'model {s_atom.model} and index {s_atom.index}',
                         f'model {t_atom.model} and index {t_atom.index}')
            if mode.dash_color:
                cmd.set('dash_color', mode.dash_color, dash_name)
            rows.append(contact_row(mode.group, filter_mode, cutoff, source_key,
                                    target_key, s_atom, t_atom, dist2))
            if make_selection:
                selection_atoms.extend((a.model, a.index)
                                       for a in target_residues[target_key])

        if verbose:
            print('Source:', s_resn, s_resi, 'object:', s_model, 'segi:',
                  repr(s_segi), 'chain:', repr(s_chain), 'contacts:', len(hits))

    if make_selection and selection_atoms:
        unique = sorted(set(selection_atoms))
        cmd.select(selection_name,
                   ' or '.join(f'(model {m} and index {i})' for m, i in unique))
        print(f"Created selection '{selection_name}' with {len(unique)} atoms.")
    elif make_selection:
        print(f"No atoms found for selection '{selection_name}'.")

    if csv_path:
        write_contact_csv(csv_path, rows)
    return rows


def inter_segi_contacts(selection, cutoff=4.0, out_prefix='inter_contacts',
                        source_sidechain_only=True, target_sidechain_only=False,
                        exclude_h=True, same_model_only=True, inter_mode='segi',
                        one_object=False, verbose=True, make_selection=True,
                        selection_name='inter_contact_residues', csv_path=None):
    """Contacts from each source residue's side chain to a DIFFERENT segi/chain."""
    return _search(INTER, selection, cutoff, out_prefix, source_sidechain_only,
                   target_sidechain_only, exclude_h, same_model_only,
                   inter_mode, one_object, verbose, make_selection,
                   selection_name, csv_path)


def intra_chain_contacts(selection, cutoff=4.0, out_prefix='intra_contacts',
                         source_sidechain_only=True, target_sidechain_only=False,
                         exclude_h=True, same_model_only=True, intra_mode='segi',
                         one_object=False, verbose=True, make_selection=True,
                         selection_name='intra_contact_residues', csv_path=None):
    """Contacts from each source residue's side chain within the SAME subunit.

    ``intra_mode`` defaults to ``'segi'``. Keying on the PDB chain letter
    instead would be wrong for these models: chains LE, ME and NE all carry
    chain letter F while belonging to three different dimers, so contacts
    between them would satisfy both this filter and the inter filter and be
    written twice, with contradictory labels.
    """
    return _search(INTRA, selection, cutoff, out_prefix, source_sidechain_only,
                   target_sidechain_only, exclude_h, same_model_only,
                   intra_mode, one_object, verbose, make_selection,
                   selection_name, csv_path)


def chain_contacts(selection, cutoff=4.0, inter_out_prefix='inter_contacts',
                   intra_out_prefix='intra_contacts', source_sidechain_only=True,
                   inter_target_sidechain_only=False,
                   intra_target_sidechain_only=False, exclude_h=True,
                   same_model_only=True, inter_mode='segi', intra_mode='segi',
                   one_object=False, verbose=True, make_selection=True,
                   inter_selection_name='inter_contact_residues',
                   intra_selection_name='intra_contact_residues',
                   csv_path='chain_contacts.csv'):
    """Both searches, inter rows first, then intra.

    ``intra_mode`` defaults to ``'segi'`` so that the two searches are disjoint
    and every contact is written exactly once.
    """
    rows = []
    rows += inter_segi_contacts(selection, cutoff, inter_out_prefix,
                                source_sidechain_only, inter_target_sidechain_only,
                                exclude_h, same_model_only, inter_mode,
                                one_object, verbose, make_selection,
                                inter_selection_name, None) or []
    rows += intra_chain_contacts(selection, cutoff, intra_out_prefix,
                                 source_sidechain_only, intra_target_sidechain_only,
                                 exclude_h, same_model_only, intra_mode,
                                 one_object, verbose, make_selection,
                                 intra_selection_name, None) or []
    if csv_path:
        written = write_contact_csv(csv_path, rows)
        if written != len(rows):
            raise AssertionError(f'row count drift: built {len(rows)}, wrote {written}')
    return rows


try:                                    # only when imported inside PyMOL
    from pymol import cmd as _cmd
    _cmd.extend('inter_segi_contacts', inter_segi_contacts)
    _cmd.extend('intra_chain_contacts', intra_chain_contacts)
    _cmd.extend('chain_contacts', chain_contacts)
except ImportError:
    pass

In [ ]:
if "." not in sys.path:
    sys.path.insert(0, ".")
import capsid
print("analysis package ready:", ", ".join(capsid.__all__))

## Detect contacts

Loads the structure, applies the selection, and writes the contact table.

In [ ]:
from capsid import pymol_contacts as pc

pdb = PDB_FILE or find_input("the structure to analyse", ["*.pdb", "*.cif"],
                             ["data/models", "data", "."])
require(pdb, "the structure to analyse")

out_csv = os.path.join(OUTPUT_DIR, "contacts.csv")

with pymol2.PyMOL() as session:
    cmd = session.cmd
    sys.modules["pymol"].cmd = cmd     # the module binds pymol.cmd on import
    cmd.load(pdb, OBJECT_NAME)
    cmd.select("Sources", SELECTION)
    print("atoms in selection:", cmd.count_atoms("Sources"))
    rows = pc.chain_contacts(selection="Sources", cutoff=CUTOFF,
                             intra_mode=INTRA_MODE, verbose=False,
                             csv_path=out_csv)

groups = {}
for row in rows:
    groups[row["contact_group"]] = groups.get(row["contact_group"], 0) + 1

identities = {(r["source_segi"], r["source_resi"], r["source_atom"],
               r["target_segi"], r["target_resi"], r["target_atom"]) for r in rows}
print(f"\nrows written      : {len(rows)}")
print(f"distinct contacts : {len(identities)}")
if len(identities) != len(rows):
    print(f"  {len(rows) - len(identities)} contact(s) written twice -- expected "
          f"with intra_mode='chain'")
print(f"by group          : {groups}")

deliver(out_csv)